# Exploratory Data Analysis

Before building any model, we explore the dataset to understand its structure, identify patterns, and spot potential challenges such as missing metadata or skewed distributions.

## Step 1: Load Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

interactions = pd.read_csv('../data/interactions_train.csv')
items        = pd.read_csv('../data/items.csv')

print(f'Total interactions: {len(interactions)}')
print(f'Total unique users:  {interactions["u"].nunique()}')
print(f'Total unique books:  {items["i"].nunique()}')


## Step 2: User Activity Distribution

How many books does a typical user borrow? A heavily skewed distribution means most users have very few interactions — a classic challenge for collaborative filtering.

In [ ]:
user_activity = interactions.groupby('u').size()

plt.figure(figsize=(10, 5))
sns.histplot(user_activity, bins=50, color='steelblue', kde=False)
plt.title('User Activity: Books Borrowed per User')
plt.xlabel('Number of Borrowed Books')
plt.ylabel('Number of Users')
plt.xlim(0, 50)
plt.tight_layout()
plt.show()

print(f'Median interactions per user: {user_activity.median():.0f}')
print(f'Mean interactions per user:   {user_activity.mean():.1f}')
print(f'Max interactions per user:    {user_activity.max()}')


## Step 3: Item Popularity — Long Tail

Most books are borrowed very rarely, while a small number of titles dominate. This **long-tail distribution** is typical in library and e-commerce data. Pure popularity-based recommendations would miss the majority of the catalogue.

In [ ]:
item_popularity = interactions.groupby('i').size().sort_values(ascending=False).values

plt.figure(figsize=(10, 5))
plt.plot(item_popularity, color='crimson')
plt.fill_between(range(len(item_popularity)), item_popularity, color='crimson', alpha=0.3)
plt.title('Item Popularity (Long Tail Distribution)')
plt.xlabel('Book Index (most → least popular)')
plt.ylabel('Number of Interactions')
plt.tight_layout()
plt.show()

print(f'Books with only 1 interaction: {(item_popularity == 1).sum()} '
      f'({100*(item_popularity==1).mean():.1f}%)')


## Step 4: Missing Metadata

Content-based filtering relies on book metadata (title, author, subjects). Missing values reduce the quality of TF-IDF representations and motivated our data enrichment efforts.

In [ ]:
missing_pct = (items.isnull().sum() / len(items) * 100).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=missing_pct.index, y=missing_pct.values, palette='viridis')
plt.title('Missing Values in Book Metadata (%)')
plt.ylabel('% Missing')
plt.tight_layout()
plt.show()

print(missing_pct.to_string())


## Step 5: Interaction Matrix — Synthetic Data Artifact

Visualising which users interact with which books reveals an unusual pattern: a **smooth diagonal frontier** separating active and inactive regions. This is highly atypical of real library data and strongly suggests the dataset is **synthetically generated**.

Users with higher IDs interact with a broader range of books across the full catalogue, while lower-ID users are confined to a smaller subset. We chose not to exploit this boundary to keep our model generalisable to real-world scenarios.

In [ ]:
# Sample a subset for visualisation (full matrix is 7838 × 15291)
sample_users = sorted(df['u'].unique())[::8]  # every 8th user
sample_df = df[df['u'].isin(sample_users)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(sample_df['i'], sample_df['u'], s=0.3, alpha=0.4, color='steelblue')
ax.set_xlabel('Book ID')
ax.set_ylabel('User ID')
ax.set_title('Interaction Matrix (sampled) — Smooth Frontier Visible')
plt.tight_layout()
plt.show()

density = len(df) / (df['u'].nunique() * items['i'].nunique())
print(f'Matrix density: {100*density:.3f}% (sparsity: {100*(1-density):.2f}%)')


## Step 6: Reader Loyalty — Author Preference

Do users tend to read multiple books by the same author, or explore broadly?

We computed the average number of books read per author for each active user (≥5 interactions). The result shows a strongly right-skewed distribution:
- Most users are **pure explorers** — reading on average only 1 book per author
- **20.6% are loyal fans** — averaging 2+ books by the same author

This motivated our **Author×2** weighting in TF-IDF: doubling the author field rewards author loyalty without ignoring subject diversity.

In [ ]:
interactions_with_author = df.merge(items[['i','Author']], on='i')
interactions_with_author = interactions_with_author[
    interactions_with_author['Author'].notna() &
    (interactions_with_author['Author'] != '')
]
active_users = df.groupby('u').size()
active_users = active_users[active_users >= 5].index
loyal = interactions_with_author[interactions_with_author['u'].isin(active_users)]
author_per_user = loyal.groupby(['u','Author']).size().reset_index(name='count')
avg_per_author = author_per_user.groupby('u')['count'].mean()

plt.figure(figsize=(10, 5))
sns.histplot(avg_per_author, bins=40, color='mediumpurple', kde=False)
plt.title('Reader Loyalty: Average Books Read per Author (active users)')
plt.xlabel('Average books per author')
plt.ylabel('Number of Users')
plt.xlim(0, 8)
plt.tight_layout()
plt.show()

loyal_pct = (avg_per_author >= 2).sum() / len(avg_per_author)
print(f'Loyal fans (avg >= 2 books/author): {100*loyal_pct:.1f}%')
print(f'Pure explorers (avg < 2): {100*(1-loyal_pct):.1f}%')


## Step 7: Repeat Borrowing

Some users borrow the same book more than once — indicating strong affinity. **16.2% of unique (user, book) pairs** have 2+ borrowings.

This signal motivated a **repeat score** experiment: boosting books that a user has already borrowed multiple times. However, in 5-fold CV it yielded only a marginal improvement (+0.0003) and did not improve the Kaggle score.

In [ ]:
repeat_counts = df.groupby(['u','i']).size().reset_index(name='count')
repeat_rate = (repeat_counts['count'] > 1).mean()
print(f'(user, book) pairs borrowed 2+ times: {100*repeat_rate:.1f}%')

plt.figure(figsize=(10, 4))
repeat_counts['count'].clip(upper=6).value_counts().sort_index().plot(
    kind='bar', color='coral', edgecolor='white')
plt.title('Borrowing Count Distribution per (User, Book) Pair')
plt.xlabel('Number of times borrowed (capped at 6)')
plt.ylabel('Number of (user, book) pairs')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Step 8: Temporal Distribution of Borrowings

Are borrowings evenly spread across the year, or do they follow a seasonal pattern?

This question matters for our evaluation strategy: if Kaggle holds out the **most recent** interactions, then a temporal split (test on last months) would be the right CV approach. If Kaggle holds out **random** interactions across all time, then a 5-fold average is better.

In [ ]:
df['date'] = pd.to_datetime(interactions['t'], unit='s')
df['month'] = df['date'].dt.to_period('M')
monthly = df.groupby('month').size()

plt.figure(figsize=(12, 4))
monthly.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Borrowings per Month (Jan 2023 – Oct 2024)')
plt.xlabel('Month')
plt.ylabel('Number of Interactions')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Monthly stats (2023 only — full year):')
y2023 = monthly[monthly.index.year == 2023]
print(f'  Min: {y2023.min():,} | Max: {y2023.max():,} | Std: {y2023.std():.0f}')


**Interpretation:**

- **2023** (full year): borrowings range from ~3,500 to ~5,900 per month — no strong seasonal peak, roughly uniform across the year.
- **2024**: interactions drop sharply toward October. This is a **data collection artifact** — the dataset ends in October 2024, so the last months are naturally incomplete.

The absence of strong seasonality in 2023 supports our hypothesis that the **Kaggle test set is a random holdout across time** (not a recent-month holdout). This was confirmed by testing seasonal weighting: boosting autumn/winter interactions produced no improvement, whereas **5-fold CV averaging outperformed last-fold-only** evaluation as a proxy for Kaggle performance.

## Key Findings

| Observation | Implication for modelling |
|---|---|
| 69.1% of users have <10 interactions | CF struggles with sparse users → need content + graph signals |
| 99.93% matrix sparsity | Hybrid approach essential |
| Long-tail popularity (top 5% books = 23.7% of interactions) | Log-normalize popularity to avoid bias |
| 17.4% missing Author, 14.5% missing Subjects | Enrichment via APIs; TF-IDF still works with gaps |
| Synthetic data artifact (smooth frontier) | Do not exploit ID proximity |
| 20.6% loyal fans (same author 2+ times) | Author×2 weighting in TF-IDF |
| 16.2% repeat borrowings | Repeat signal tested but gave marginal improvement |
